# Spark Internals — DAG, Shuffle, Memory, Catalyst

## Mental Model

Spark turns your code into a **DAG of transformations** and then breaks that DAG into **stages** separated by shuffles.

The big tuning levers are usually:

- minimize shuffles
- push filters down early
- size partitions sanely
- let Catalyst and AQE optimize, but verify with plans

Citi framing used throughout this notebook:

- 6,000+ API endpoints monitored for latency, throughput, and error rate
- 500K metric records and 25K alerts provide enough volume to show realistic Spark mechanics
- local `master=local[*]` is enough to demonstrate the internals that matter in production reasoning


In [ ]:
import io
import os
from contextlib import redirect_stdout

os.environ['JAVA_HOME'] = r'C:/Program Files/Java/jre1.8.0_481'
os.environ['HADOOP_HOME'] = r'C:/hadoop'

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

JDBC_URL = 'jdbc:postgresql://localhost:5432/de_telemetry'
JDBC_PROPERTIES = {
    'user': 'de_admin',
    'password': 'DeAdmin2026!',
    'driver': 'org.postgresql.Driver',
}

spark = (
    SparkSession.builder
    .appName('spark_internals_tuning')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '10')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory', '2g')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')

def explain_str(df, extended=True):
    buf = io.StringIO()
    with redirect_stdout(buf):
        df.explain(extended)
    return buf.getvalue()

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"JAVA_HOME: {os.environ['JAVA_HOME']}")
print(f"HADOOP_HOME: {os.environ['HADOOP_HOME']}")
print(f"spark.executor.memory: {spark.conf.get('spark.executor.memory')}")
print(f"spark.sql.adaptive.enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")


## DAG Inspection

We load `endpoints` and `alerts` via JDBC, join them on `endpoint_id`, and aggregate alert counts by region and severity.

What to look for in the plan:

- JDBC scans as the leaf nodes
- a join operator in the middle of the DAG
- an aggregation above the join
- one or more exchanges indicating shuffle boundaries

Those exchange operators are the places where Spark breaks the job into stages.


In [ ]:
endpoints_df = (
    spark.read.format('jdbc')
    .option('url', JDBC_URL)
    .option('dbtable', 'endpoints')
    .option('user', JDBC_PROPERTIES['user'])
    .option('password', JDBC_PROPERTIES['password'])
    .option('driver', JDBC_PROPERTIES['driver'])
    .load()
)

alerts_df = (
    spark.read.format('jdbc')
    .option('url', JDBC_URL)
    .option('dbtable', 'alerts')
    .option('user', JDBC_PROPERTIES['user'])
    .option('password', JDBC_PROPERTIES['password'])
    .option('driver', JDBC_PROPERTIES['driver'])
    .load()
)

dag_df = (
    alerts_df.alias('a')
    .join(endpoints_df.alias('e'), F.col('a.endpoint_id') == F.col('e.endpoint_id'), 'inner')
    .groupBy(F.col('e.region').alias('region'), F.col('a.severity').alias('severity'))
    .agg(F.count('*').alias('alert_count'))
    .orderBy(F.desc('alert_count'))
)

dag_plan = explain_str(dag_df, extended=True)
print(dag_plan)
dag_df.show(10, truncate=False)


### DAG Reading Notes

The logical plan describes *what* Spark needs to do, while the physical plan shows *how* Spark will do it.

In this query:

- JDBC scans read `alerts` and `endpoints`
- the join combines endpoint metadata with alert facts
- `groupBy(region, severity)` forces aggregation work
- an `Exchange` node marks a shuffle, which separates stages

For Staff-level interviews, that is the key sentence: **wide transformations create stage boundaries because data has to move across partitions.**


## Shuffle Deep Dive

Now we force a wide transformation by repartitioning the joined data to 10 partitions and inspecting row distribution.

The point is not just to move data, but to make the shuffle visible as a cost center:

- shuffle write: map-side tasks write partitioned intermediate data
- shuffle read: downstream tasks fetch the needed blocks over the network or local disk
- skew or tiny partitions both hurt efficiency in different ways


In [ ]:
shuffle_base_df = (
    alerts_df.select('alert_id', 'endpoint_id', 'severity')
    .join(endpoints_df.select('endpoint_id', 'region'), on='endpoint_id', how='inner')
)

repartitioned_df = shuffle_base_df.repartition(10, 'region')
partition_sizes = repartitioned_df.rdd.glom().map(len).collect()
print('Partition distribution after repartition(10, region):')
for idx, size in enumerate(partition_sizes):
    print(f'Partition {idx}: {size} rows')

partition_count_df = repartitioned_df.groupBy(F.spark_partition_id().alias('partition_id')).count().orderBy('partition_id')
partition_count_df.show(20, truncate=False)


### Shuffle Cost Notes

A shuffle is expensive because Spark materializes intermediate results, redistributes them by key, and then reconstructs downstream tasks from those remote blocks.

That means shuffle-heavy jobs are usually bottlenecked by some combination of:

- network transfer
- serialization / deserialization
- disk spill and disk I/O
- skewed tasks waiting on one oversized partition

At Citi scale, the fastest Spark jobs are often the ones that avoided an unnecessary shuffle in the first place.


## Memory Management

We use the 500K-row `metrics` table to force heavier processing and inspect Spark memory settings.

This is still local mode, so you should read the lesson conceptually:

- **execution memory** powers joins, aggregations, sorts, and shuffles
- **storage memory** keeps cached data
- **on-heap** is normal JVM memory
- **off-heap** can be enabled separately for advanced tuning

When execution memory pressure rises, Spark may spill intermediate data to disk. Spill is not failure; it is Spark choosing slower I/O over OOM.


In [ ]:
metrics_df = (
    spark.read.format('jdbc')
    .option('url', JDBC_URL)
    .option('dbtable', 'metrics')
    .option('user', JDBC_PROPERTIES['user'])
    .option('password', JDBC_PROPERTIES['password'])
    .option('driver', JDBC_PROPERTIES['driver'])
    .load()
)

print(f"metrics row count: {metrics_df.count()}")
print(f"spark.executor.memory: {spark.conf.get('spark.executor.memory')}")
print(f"spark.memory.fraction: {spark.conf.get('spark.memory.fraction', '0.6 (default)')}")
print(f"spark.memory.storageFraction: {spark.conf.get('spark.memory.storageFraction', '0.5 (default)')}")
print(f"spark.memory.offHeap.enabled: {spark.conf.get('spark.memory.offHeap.enabled', 'false')}")
print(f"spark.memory.offHeap.size: {spark.conf.get('spark.memory.offHeap.size', '0')}")

spill_demo_df = (
    metrics_df
    .repartition(10, 'metric_name')
    .groupBy('metric_name', 'endpoint_id')
    .agg(
        F.avg('value').alias('avg_value'),
        F.max('value').alias('max_value'),
        F.count('*').alias('samples')
    )
    .orderBy(F.desc('samples'), F.desc('avg_value'))
)

spill_demo_df.show(10, truncate=False)
print(explain_str(spill_demo_df, extended=False))


### Memory Notes

Even if local mode does not visibly scream about spills in notebook output, this workload still demonstrates the kind of aggregation and ordering pattern that consumes execution memory.

The interview-level explanation is:

- Spark divides unified memory between execution and storage
- cached data competes with shuffle / aggregation work
- when execution needs more room, Spark may evict storage or spill intermediate structures to disk

That is why caching everything and then complaining about slow joins is such a classic Spark anti-pattern.


## Catalyst Optimizer

We compare two patterns:

1. filter **before** join
2. filter **after** join

Then we compare automatic versus manual broadcast behavior.

The key lesson: **predicate pushdown and join strategy selection often matter more than micro-tweaks.**


In [ ]:
filter_before_join_df = (
    alerts_df.filter(F.col('severity').isin('HIGH', 'CRITICAL'))
    .join(endpoints_df.filter(F.col('region') == 'us-east-1'), on='endpoint_id', how='inner')
    .select('endpoint_id', 'severity', 'region')
)

filter_after_join_df = (
    alerts_df.join(endpoints_df, on='endpoint_id', how='inner')
    .filter((F.col('severity').isin('HIGH', 'CRITICAL')) & (F.col('region') == 'us-east-1'))
    .select('endpoint_id', 'severity', 'region')
)

print('Plan: filter before join')
print(explain_str(filter_before_join_df, extended=False))
print('Plan: filter after join')
print(explain_str(filter_after_join_df, extended=False))

print(f"autoBroadcastJoinThreshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')}")

auto_join_df = alerts_df.join(endpoints_df.select('endpoint_id', 'region', 'category'), on='endpoint_id', how='inner')
manual_broadcast_df = alerts_df.join(broadcast(endpoints_df.select('endpoint_id', 'region', 'category')), on='endpoint_id', how='inner')

print('Plan: automatic join strategy')
print(explain_str(auto_join_df, extended=False))
print('Plan: manual broadcast() hint')
print(explain_str(manual_broadcast_df, extended=False))


### Catalyst Notes

Catalyst applies rule-based and cost-aware rewrites that can:

- push filters toward data sources
- prune unused columns
- reorder or simplify expressions
- choose a better join strategy

Manual `broadcast()` is useful when you know a dimension table is small enough and want to force a broadcast hash join instead of waiting for automatic selection.


## AQE (Adaptive Query Execution)

AQE lets Spark change parts of the physical plan after runtime statistics become available.

Here we compare a skew-oriented join pattern with AQE off and AQE on.

The point is to show the plan difference, not to pretend local mode perfectly reproduces a full production cluster skew scenario.


In [ ]:
skewed_alerts_df = alerts_df.withColumn(
    'skew_key',
    F.when(F.col('severity').isin('HIGH', 'CRITICAL'), F.lit('hot')).otherwise(F.lit('cold'))
)
skewed_endpoints_df = endpoints_df.withColumn(
    'skew_key',
    F.when(F.col('status') == 'active', F.lit('hot')).otherwise(F.lit('cold'))
)

spark.conf.set('spark.sql.adaptive.enabled', 'false')
aqe_off_df = (
    skewed_alerts_df.join(skewed_endpoints_df.select('endpoint_id', 'region', 'skew_key'), on=['endpoint_id', 'skew_key'], how='inner')
    .groupBy('skew_key', 'region')
    .count()
)
aqe_off_df.count()
aqe_off_plan = explain_str(aqe_off_df, extended=False)

spark.conf.set('spark.sql.adaptive.enabled', 'true')
aqe_on_df = (
    skewed_alerts_df.join(skewed_endpoints_df.select('endpoint_id', 'region', 'skew_key'), on=['endpoint_id', 'skew_key'], how='inner')
    .groupBy('skew_key', 'region')
    .count()
)
aqe_on_df.count()
aqe_on_plan = explain_str(aqe_on_df, extended=False)

print('Plan with AQE disabled:')
print(aqe_off_plan)
print('Plan with AQE enabled:')
print(aqe_on_plan)


### AQE Notes

AQE can do things like:

- coalesce tiny shuffle partitions
- switch join strategies at runtime
- mitigate skew by splitting oversized partitions

In a full distributed environment, AQE is one of the highest-leverage Spark settings because it reacts to the data shape you *actually* had, not just what the optimizer guessed upfront.


## What Just Happened

Every Spark job is a DAG of stages separated by shuffles. Tuning = minimize shuffles, maximize pushdown, right-size partitions. Citi Spark jobs on 500K metrics + 25K alerts should run in under 10 seconds on `local[*]`.

The Staff-level summary is simple:

- **DAG** tells you what work exists
- **Exchange / shuffle** tells you where the expensive boundaries are
- **memory tuning** determines whether heavy operations stay in memory or spill
- **Catalyst + AQE** often fix more than manual tweaking, but only if you verify the plan
